# Notebook 04: Attention Analysis (Level 3)

## Overview
Analyzes **attention patterns** from the prediction position across all layers and heads
in Pythia models performing the LSC induction task. Characterizes where each head attends,
classifies head roles (induction, previous-token, BOS sink, diffuse), and examines how
these patterns relate to model accuracy and frequency band structure.

## Key Questions
- Where does each head attend from the prediction position?
- Which heads are induction heads, BOS sinks, previous-token, or diffuse?
- Does head role classification change across frequency bands?
- Do induction heads show different attention distributions for correct vs incorrect predictions?
- Which heads have the highest copy scores (OV circuit for copying)?
- How does the attention sink phenomenon manifest across layers?

## Hypothesis Domain: R4 (Attention Patterns)
- **H-R4.1**: Induction heads attend strongly to the induction cue position
- **H-R4.2**: Head role classification is stable across frequency bands
- **H-R4.3**: Induction heads show higher induction scores for correctly predicted examples
- **H-R4.4**: BOS attention fraction decreases in later layers (attention sink is early)
- **H-R4.5**: Copy scores correlate with induction scores from attention patterns
- **H-R4.6**: Larger models have more specialized (less diffuse) heads

## Notebook Structure
1. Setup & Data Loading
2. Positional Attention Statistics
3. Attention Entropy
4. Head Role Classification
5. Head Role Stability Across Bands
6. Copy Score Analysis
7. Attention Sink Analysis
8. Induction Pattern Analysis (Correct vs Incorrect)
9. Cross-Model Comparison
10. Summary

## Data Sources
- Pre-extracted activations: `attn_pattern_predpos` from NPZ files, shape (N, n_layers, n_heads, 22)
- Also: `target_ids` (N,), `input_ids` (N, 21)
- Pre-computed copy scores: `copy_scores_{model_dir}.npz`
- Logit lens metrics: `logit_lens_prob_correct` from NPZ files (for correct/incorrect split)

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_PREDICTION_POS,
    MODEL_SOURCE_POS,
    MODEL_TARGET_POS,
    MODEL_REPEAT_POS,
    MODEL_INDUCTION_CUE_POS,
    SEQ_LEN_WITH_BOS,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    load_copy_scores,
    save_analysis,
)
from utils.attention import (
    compute_attention_entropy,
    compute_positional_attention_stats,
    classify_head_role,
    classify_all_heads,
)
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_head_role_map,
    plot_metric_heatmap,
    plot_boxplot_by_band,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("attention", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Prediction position (with BOS): {MODEL_PREDICTION_POS}")
print(f"Induction cue position (with BOS): {MODEL_INDUCTION_CUE_POS}")
print(f"Sequence length (with BOS): {SEQ_LEN_WITH_BOS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Prediction position (with BOS): 21
Induction cue position (with BOS): 5
Sequence length (with BOS): 22


In [2]:
# Load attention patterns at prediction position for all models/bands/draws
# Shape per file: attn_pattern_predpos = (N, n_layers, n_heads, 22)
all_attn = {}  # model -> draw -> band -> (N, n_layers, n_heads, 22)
all_targets = {}  # model -> draw -> band -> (N,)
all_inputs = {}  # model -> draw -> band -> (N, 21)
all_logit_lens_prob = {}  # model -> draw -> band -> (N, n_layers)

for model in MODELS:
    all_attn[model] = {}
    all_targets[model] = {}
    all_inputs[model] = {}
    all_logit_lens_prob[model] = {}
    for draw in DRAWS:
        all_attn[model][draw] = {}
        all_targets[model][draw] = {}
        all_inputs[model][draw] = {}
        all_logit_lens_prob[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                all_attn[model][draw][band] = data["attn_pattern_predpos"]
                if "target_ids" in data:
                    all_targets[model][draw][band] = data["target_ids"]
                if "input_ids" in data:
                    all_inputs[model][draw][band] = data["input_ids"]
                if "logit_lens_prob_correct" in data:
                    all_logit_lens_prob[model][draw][band] = data[
                        "logit_lens_prob_correct"
                    ]
            except FileNotFoundError:
                pass

# Report shapes
for model in MODELS:
    sample = next(iter(next(iter(all_attn[model].values())).values()), None)
    if sample is not None:
        n_layers = MODEL_INFO[model]["n_layers"]
        n_heads = MODEL_INFO[model]["n_heads"]
        print(
            f"{model}: attn shape = {sample.shape} "
            f"(N, {n_layers} layers, {n_heads} heads, {SEQ_LEN_WITH_BOS} positions)"
        )

pythia-70m: attn shape = (225, 6, 8, 22) (N, 6 layers, 8 heads, 22 positions)
pythia-160m: attn shape = (225, 12, 12, 22) (N, 12 layers, 12 heads, 22 positions)
pythia-410m: attn shape = (225, 24, 16, 22) (N, 24 layers, 16 heads, 22 positions)
pythia-1b: attn shape = (225, 16, 8, 22) (N, 16 layers, 8 heads, 22 positions)
pythia-1.4b: attn shape = (225, 24, 16, 22) (N, 24 layers, 16 heads, 22 positions)


## 2. Positional Attention Statistics

From the prediction position, compute the fraction of attention allocated to each
semantic region: BOS (pos 0), source (pos 1-5), target (pos 6), distractors (pos 7-16),
repeat (pos 17-21), and the induction cue (pos 16). Statistics computed per head and per band.

In [3]:
# Compute positional attention statistics for all models/bands/draws
pos_attn_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    print(f"\nProcessing {model} ({n_layers}L x {n_heads}H)...")

    for draw in DRAWS:
        for band in BANDS:
            attn = all_attn.get(model, {}).get(draw, {}).get(band)
            if attn is None:
                continue

            # attn shape: (N, n_layers, n_heads, seq_len_with_bos)
            stats = compute_positional_attention_stats(attn)
            # Each stat is (N, n_layers, n_heads): take mean over examples

            for layer in range(n_layers):
                for head in range(n_heads):
                    record = {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "head": head,
                    }
                    for stat_name, stat_vals in stats.items():
                        record[stat_name] = float(stat_vals[:, layer, head].mean())
                        record[f"{stat_name}_std"] = float(
                            stat_vals[:, layer, head].std()
                        )
                    pos_attn_records.append(record)

df_pos_attn = pd.DataFrame(pos_attn_records)
save_analysis(df_pos_attn, "04_positional_attention_stats.csv")
print(f"\nPositional attention records: {len(df_pos_attn)}")
print(f"Columns: {list(df_pos_attn.columns)}")


Processing pythia-70m (6L x 8H)...



Processing pythia-160m (12L x 12H)...



Processing pythia-410m (24L x 16H)...



Processing pythia-1b (16L x 8H)...



Processing pythia-1.4b (24L x 16H)...



Positional attention records: 16320
Columns: ['model', 'draw', 'band', 'layer', 'head', 'bos_fraction', 'bos_fraction_std', 'self_fraction', 'self_fraction_std', 'prev_fraction', 'prev_fraction_std', 'source_fraction', 'source_fraction_std', 'target_fraction', 'target_fraction_std', 'repeat_fraction', 'repeat_fraction_std', 'induction_score', 'induction_score_std', 'distractor_fraction', 'distractor_fraction_std', 'entropy', 'entropy_std']


In [4]:
# Visualize: mean attention to each region per layer (draw_1 only, averaged over heads)
regions = [
    "bos_fraction",
    "source_fraction",
    "target_fraction",
    "distractor_fraction",
    "repeat_fraction",
    "induction_score",
]
region_labels = [
    "BOS",
    "Source (S1-S5)",
    "Target (T)",
    "Distractors",
    "Repeat (S1-S5)",
    "Induction Cue",
]
region_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]

for model in MODELS:
    model_data = df_pos_attn[
        (df_pos_attn["model"] == model) & (df_pos_attn["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    n_bands = len(BANDS)
    fig, axes = plt.subplots(1, n_bands, figsize=(5 * n_bands, 5), sharey=True)
    if n_bands == 1:
        axes = [axes]

    for ax, band in zip(axes, BANDS):
        band_data = model_data[model_data["band"] == band]
        if len(band_data) == 0:
            continue

        # Average over heads per layer
        layer_means = band_data.groupby("layer")[regions].mean()

        for region, label, color in zip(regions, region_labels, region_colors):
            if region in layer_means.columns:
                ax.plot(
                    layer_means.index,
                    layer_means[region],
                    label=label,
                    color=color,
                    marker="o",
                    markersize=3,
                )

        ax.set_xlabel("Layer")
        ax.set_title(BAND_NAMES.get(band, band))

    axes[0].set_ylabel("Mean Attention Fraction")
    axes[0].legend(fontsize=7, loc="upper left")
    fig.suptitle(f"Positional Attention by Region: {model}", y=1.02)
    fig.tight_layout()
    save_figure(fig, f"viz_04_01_positional_attention_{model}.png")

## 3. Attention Entropy (Bits)

Entropy of the attention distribution from the prediction position. Low entropy = concentrated
attention (specialized heads). High entropy = diffuse attention.

In [5]:
# Compute attention entropy per head, per band
entropy_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for draw in DRAWS:
        for band in BANDS:
            attn = all_attn.get(model, {}).get(draw, {}).get(band)
            if attn is None:
                continue

            # Compute entropy per example, per layer, per head
            # attn shape: (N, n_layers, n_heads, seq_len_with_bos)
            entropy = compute_attention_entropy(attn)  # (N, n_layers, n_heads)

            for layer in range(n_layers):
                for head in range(n_heads):
                    head_entropy = entropy[:, layer, head]
                    entropy_records.append(
                        {
                            "model": model,
                            "draw": draw,
                            "band": band,
                            "layer": layer,
                            "head": head,
                            "entropy_mean": float(head_entropy.mean()),
                            "entropy_std": float(head_entropy.std()),
                            "entropy_median": float(np.median(head_entropy)),
                        }
                    )

df_entropy = pd.DataFrame(entropy_records)
save_analysis(df_entropy, "04_attention_entropy.csv")
print(f"Entropy records: {len(df_entropy)}")
print(f"Max entropy for 22 positions = {np.log2(SEQ_LEN_WITH_BOS):.2f} bits")

Entropy records: 16320
Max entropy for 22 positions = 4.46 bits


In [6]:
# Visualize: entropy heatmap per model (head x layer, draw_1, averaged over bands)
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    model_data = df_entropy[
        (df_entropy["model"] == model) & (df_entropy["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    # Average over bands to get overall head characterization
    avg_data = (
        model_data.groupby(["layer", "head"])["entropy_mean"].mean().reset_index()
    )
    pivot = avg_data.pivot(index="head", columns="layer", values="entropy_mean")

    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
    sns.heatmap(
        pivot,
        annot=n_heads <= 16,
        fmt=".2f",
        cmap="viridis",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Attention Entropy (bits): {model}")
    save_figure(fig, f"viz_04_02_entropy_heatmap_{model}.png")

In [7]:
# Entropy distribution across layers (mean over heads, draw_1)
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, MODELS):
    model_data = df_entropy[
        (df_entropy["model"] == model) & (df_entropy["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    for band in BANDS:
        bd = model_data[model_data["band"] == band]
        if len(bd) == 0:
            continue
        layer_means = bd.groupby("layer")["entropy_mean"].mean()
        ax.plot(
            layer_means.index,
            layer_means.values,
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )

    ax.set_xlabel("Layer")
    ax.set_title(model)
    ax.axhline(
        y=np.log2(SEQ_LEN_WITH_BOS),
        color="gray",
        linestyle="--",
        alpha=0.3,
        label="Max entropy",
    )

axes[0].set_ylabel("Mean Entropy (bits)")
axes[0].legend(fontsize=7)
fig.suptitle("Attention Entropy Across Layers (avg over heads)", y=1.02)
fig.tight_layout()
save_figure(fig, "viz_04_03_entropy_by_layer.png")

## 4. Head Role Classification

Classify each head as: **induction**, **previous_token**, **bos_sink**, or **diffuse**
based on mean positional attention statistics. Uses dominance-ratio criterion from
`classify_all_heads`.

In [8]:
# Classify heads for each model (using draw_1, all bands combined)
head_role_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    print(f"\n{model}:")

    # Combine all bands from draw_1 for overall classification
    all_band_attn = []
    for band in BANDS:
        attn = all_attn.get(model, {}).get("draw_1", {}).get(band)
        if attn is not None:
            all_band_attn.append(attn)

    if len(all_band_attn) == 0:
        continue

    combined_attn = np.concatenate(
        all_band_attn, axis=0
    )  # (N_total, n_layers, n_heads, 22)

    # Classify all heads
    results = classify_all_heads(combined_attn, n_layers, n_heads)

    for r in results:
        r["model"] = model
        head_role_records.append(r)

    # Print role distribution
    roles = [r["role"] for r in results]
    from collections import Counter

    role_counts = Counter(roles)
    total = len(roles)
    for role, count in sorted(role_counts.items()):
        print(f"  {role}: {count}/{total} ({100 * count / total:.1f}%)")

df_head_roles = pd.DataFrame(head_role_records)
save_analysis(df_head_roles, "04_head_role_classification.csv")
print(f"\nHead role records: {len(df_head_roles)}")


pythia-70m:
  bos_sink: 17/48 (35.4%)
  diffuse: 17/48 (35.4%)
  induction: 3/48 (6.2%)
  previous_token: 11/48 (22.9%)

pythia-160m:
  bos_sink: 82/144 (56.9%)
  diffuse: 37/144 (25.7%)
  induction: 5/144 (3.5%)
  previous_token: 20/144 (13.9%)

pythia-410m:
  bos_sink: 241/384 (62.8%)
  diffuse: 111/384 (28.9%)
  induction: 8/384 (2.1%)
  previous_token: 24/384 (6.2%)

pythia-1b:
  bos_sink: 72/128 (56.2%)
  diffuse: 41/128 (32.0%)
  induction: 4/128 (3.1%)
  previous_token: 11/128 (8.6%)

pythia-1.4b:
  bos_sink: 258/384 (67.2%)
  diffuse: 102/384 (26.6%)
  induction: 7/384 (1.8%)
  previous_token: 17/384 (4.4%)

Head role records: 1088


In [9]:
# Visualize: head role heatmap per model
for model in MODELS:
    model_roles = df_head_roles[df_head_roles["model"] == model]
    if len(model_roles) == 0:
        continue

    fig = plot_head_role_map(model_roles, model)
    save_figure(fig, f"viz_04_04_head_roles_{model}.png")

In [10]:
# Summary table: role distribution per model
role_summary = df_head_roles.groupby(["model", "role"]).size().unstack(fill_value=0)
role_summary["total"] = role_summary.sum(axis=1)
for role in ["induction", "previous_token", "bos_sink", "diffuse"]:
    if role in role_summary.columns:
        role_summary[f"{role}_pct"] = (
            100 * role_summary[role] / role_summary["total"]
        ).round(1)

print("Head Role Distribution per Model:")
print(role_summary)
save_analysis(role_summary.reset_index(), "04_head_role_summary.csv")

Head Role Distribution per Model:
role         bos_sink  diffuse  induction  previous_token  total  \
model                                                              
pythia-1.4b       258      102          7              17    384   
pythia-160m        82       37          5              20    144   
pythia-1b          72       41          4              11    128   
pythia-410m       241      111          8              24    384   
pythia-70m         17       17          3              11     48   

role         induction_pct  previous_token_pct  bos_sink_pct  diffuse_pct  
model                                                                      
pythia-1.4b            1.8                 4.4          67.2         26.6  
pythia-160m            3.5                13.9          56.9         25.7  
pythia-1b              3.1                 8.6          56.2         32.0  
pythia-410m            2.1                 6.2          62.8         28.9  
pythia-70m             6.2       

## 5. Head Role Stability Across Bands

Does the classification of a head change when we restrict to a specific frequency band?
Stable roles suggest the mechanism is band-independent; unstable roles suggest
frequency-dependent attention routing.

In [11]:
# Classify heads per band (draw_1 only)
band_role_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for band in BANDS:
        attn = all_attn.get(model, {}).get("draw_1", {}).get(band)
        if attn is None:
            continue

        results = classify_all_heads(attn, n_layers, n_heads)
        for r in results:
            r["model"] = model
            r["band"] = band
            band_role_records.append(r)

df_band_roles = pd.DataFrame(band_role_records)
save_analysis(df_band_roles, "04_head_role_by_band.csv")
print(f"Band-specific role records: {len(df_band_roles)}")

Band-specific role records: 5440


In [12]:
# Compute stability: for each head, how many bands assign the same role?
stability_records = []

for model in MODELS:
    model_data = df_band_roles[df_band_roles["model"] == model]
    if len(model_data) == 0:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    n_bands = len(model_data["band"].unique())

    for layer in range(n_layers):
        for head in range(n_heads):
            head_data = model_data[
                (model_data["layer"] == layer) & (model_data["head"] == head)
            ]
            if len(head_data) == 0:
                continue

            roles_across_bands = head_data["role"].values
            unique_roles = set(roles_across_bands)
            # Majority role
            from collections import Counter

            role_counts = Counter(roles_across_bands)
            majority_role = role_counts.most_common(1)[0][0]
            majority_frac = role_counts.most_common(1)[0][1] / len(roles_across_bands)

            stability_records.append(
                {
                    "model": model,
                    "layer": layer,
                    "head": head,
                    "n_unique_roles": len(unique_roles),
                    "majority_role": majority_role,
                    "majority_fraction": majority_frac,
                    "is_stable": len(unique_roles) == 1,
                }
            )

df_stability = pd.DataFrame(stability_records)
save_analysis(df_stability, "04_head_role_stability.csv")

# Print summary
for model in MODELS:
    md = df_stability[df_stability["model"] == model]
    if len(md) == 0:
        continue
    stable_pct = 100 * md["is_stable"].mean()
    mean_majority = md["majority_fraction"].mean()
    print(
        f"{model}: {stable_pct:.1f}% heads perfectly stable, "
        f"mean majority fraction = {mean_majority:.3f}"
    )

pythia-70m: 77.1% heads perfectly stable, mean majority fraction = 0.921
pythia-160m: 82.6% heads perfectly stable, mean majority fraction = 0.946
pythia-410m: 87.0% heads perfectly stable, mean majority fraction = 0.957
pythia-1b: 86.7% heads perfectly stable, mean majority fraction = 0.955
pythia-1.4b: 87.2% heads perfectly stable, mean majority fraction = 0.960


In [13]:
# Visualize: head role map per band for one model (e.g., pythia-410m)
# Show how roles shift across bands
example_model = "pythia-410m" if "pythia-410m" in MODELS else MODELS[0]

n_bands_to_show = len(BANDS)
n_heads_ex = MODEL_INFO[example_model]["n_heads"]
n_layers_ex = MODEL_INFO[example_model]["n_layers"]
fig_width = max(10, n_layers_ex * 0.6 + 2) * n_bands_to_show / 2
fig_height = max(5, n_heads_ex * 0.4)

fig_roles_bands, axes_rb = plt.subplots(
    1, n_bands_to_show, figsize=(fig_width, fig_height), sharey=True
)
if n_bands_to_show == 1:
    axes_rb = [axes_rb]

role_to_int = {"induction": 3, "previous_token": 2, "bos_sink": 1, "diffuse": 0}
role_colors_list = ["#d9d9d9", "#ff9896", "#aec7e8", "#98df8a"]
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

cmap_roles = ListedColormap(role_colors_list)
norm_roles = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap_roles.N)

for ax, band in zip(axes_rb, BANDS):
    band_data = df_band_roles[
        (df_band_roles["model"] == example_model) & (df_band_roles["band"] == band)
    ]
    if len(band_data) == 0:
        continue

    pivot = band_data.pivot(index="head", columns="layer", values="role")
    numeric = pivot.map(lambda x: role_to_int.get(x, -1))

    ax.imshow(numeric.values, cmap=cmap_roles, norm=norm_roles, aspect="auto")
    ax.set_xlabel("Layer")
    ax.set_title(BAND_NAMES.get(band, band))
    ax.set_xticks(range(numeric.shape[1]))
    ax.set_xticklabels(numeric.columns)
    ax.set_yticks(range(numeric.shape[0]))
    ax.set_yticklabels(numeric.index)
    ax.grid(False)

axes_rb[0].set_ylabel("Head")
legend_elements = [
    Patch(facecolor=role_colors_list[3], label="Induction"),
    Patch(facecolor=role_colors_list[2], label="Previous Token"),
    Patch(facecolor=role_colors_list[1], label="BOS Sink"),
    Patch(facecolor=role_colors_list[0], label="Diffuse"),
]
fig_roles_bands.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.08),
    fontsize=9,
)
fig_roles_bands.suptitle(f"Head Roles by Band: {example_model}", y=1.02)
fig_roles_bands.tight_layout()
save_figure(fig_roles_bands, f"viz_04_05_head_roles_by_band_{example_model}.png")

## 6. Copy Score Analysis

Load pre-computed OV copy scores. Identify which heads have the strongest copying
mechanism and whether this varies by frequency band.

In [14]:
# Load pre-computed copy scores
copy_scores = {}

for model in MODELS:
    try:
        cs_data = load_copy_scores(model)
        copy_scores[model] = cs_data
        print(f"{model}: copy scores loaded, keys = {list(cs_data.keys())}")
        # Print shapes
        for k, v in cs_data.items():
            if isinstance(v, np.ndarray):
                print(f"  {k}: shape = {v.shape}")
    except FileNotFoundError:
        print(f"{model}: copy scores not found")

pythia-70m: copy scores loaded, keys = ['copy_scores']
  copy_scores: shape = (6, 8)
pythia-160m: copy scores loaded, keys = ['copy_scores']
  copy_scores: shape = (12, 12)
pythia-410m: copy scores loaded, keys = ['copy_scores']
  copy_scores: shape = (24, 16)
pythia-1b: copy scores loaded, keys = ['copy_scores']
  copy_scores: shape = (16, 8)
pythia-1.4b: copy scores loaded, keys = ['copy_scores']
  copy_scores: shape = (24, 16)


In [15]:
# Analyze copy scores: identify top copy heads per model
copy_head_records = []

for model in MODELS:
    cs_data = copy_scores.get(model)
    if cs_data is None:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Check for overall copy scores
    if "copy_scores" in cs_data:
        scores_matrix = cs_data["copy_scores"]  # (n_layers, n_heads)
    else:
        # Try to reconstruct from band-level scores
        band_scores = []
        for key in cs_data:
            if key.startswith("copy_scores_"):
                band_scores.append(cs_data[key])
        if band_scores:
            scores_matrix = np.mean(band_scores, axis=0)
        else:
            continue

    print(f"\n{model}: Top copy heads:")
    for layer in range(n_layers):
        for head in range(n_heads):
            score = float(scores_matrix[layer, head])
            copy_head_records.append(
                {
                    "model": model,
                    "layer": layer,
                    "head": head,
                    "copy_score": score,
                }
            )

    # Print top 5
    model_records = [r for r in copy_head_records if r["model"] == model]
    top_heads = sorted(model_records, key=lambda x: x["copy_score"], reverse=True)[:5]
    for r in top_heads:
        print(f"  L{r['layer']}H{r['head']}: copy_score = {r['copy_score']:.4f}")

df_copy = pd.DataFrame(copy_head_records)
save_analysis(df_copy, "04_copy_scores.csv")
print(f"\nTotal copy score records: {len(df_copy)}")


pythia-70m: Top copy heads:
  L3H3: copy_score = 0.1204
  L3H1: copy_score = 0.0797
  L1H4: copy_score = 0.0244
  L3H0: copy_score = 0.0226
  L3H6: copy_score = 0.0216

pythia-160m: Top copy heads:
  L5H9: copy_score = 0.0281
  L8H10: copy_score = 0.0152
  L6H11: copy_score = 0.0141
  L4H11: copy_score = 0.0128
  L7H3: copy_score = 0.0127

pythia-410m: Top copy heads:
  L19H0: copy_score = 0.0033
  L14H14: copy_score = 0.0028
  L11H7: copy_score = 0.0025
  L19H8: copy_score = 0.0020
  L16H11: copy_score = 0.0018

pythia-1b: Top copy heads:
  L15H6: copy_score = 0.0040
  L12H7: copy_score = 0.0035
  L13H6: copy_score = 0.0034
  L15H5: copy_score = 0.0031
  L9H0: copy_score = 0.0030

pythia-1.4b: Top copy heads:
  L23H11: copy_score = 0.0038
  L13H10: copy_score = 0.0038
  L19H14: copy_score = 0.0028
  L16H1: copy_score = 0.0027
  L15H13: copy_score = 0.0026

Total copy score records: 1088


In [16]:
# Visualize: copy score heatmap per model (head x layer)
for model in MODELS:
    model_copy = df_copy[df_copy["model"] == model]
    if len(model_copy) == 0:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    pivot = model_copy.pivot(index="head", columns="layer", values="copy_score")

    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
    sns.heatmap(
        pivot,
        annot=False,
        cmap="YlOrRd",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"Copy Scores (OV Circuit): {model}")
    save_figure(fig, f"viz_04_06_copy_scores_{model}.png")

In [17]:
# Band-level copy scores if available
band_copy_records = []

for model in MODELS:
    cs_data = copy_scores.get(model)
    if cs_data is None:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for band in BANDS:
        key = f"copy_scores_{band}"
        if key not in cs_data:
            continue

        band_matrix = cs_data[key]  # (n_layers, n_heads)
        for layer in range(n_layers):
            for head in range(n_heads):
                band_copy_records.append(
                    {
                        "model": model,
                        "band": band,
                        "layer": layer,
                        "head": head,
                        "copy_score": float(band_matrix[layer, head]),
                    }
                )

if band_copy_records:
    df_band_copy = pd.DataFrame(band_copy_records)
    save_analysis(df_band_copy, "04_copy_scores_by_band.csv")

    # Summary: mean copy score across all heads per model per band
    summary = df_band_copy.groupby(["model", "band"])["copy_score"].mean().unstack()
    print("Mean copy score per model per band:")
    print(summary.round(4))
else:
    print("No band-level copy scores available.")

No band-level copy scores available.


## 7. Attention Sink Analysis

Examine BOS attention fraction across layers and heads. The "attention sink"
phenomenon means the BOS token absorbs excess attention probability,
especially in early layers.

In [18]:
# BOS attention fraction analysis (from df_pos_attn, draw_1)
bos_records = []

for model in MODELS:
    model_data = df_pos_attn[
        (df_pos_attn["model"] == model) & (df_pos_attn["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    # Average over bands for overall characterization
    avg_bos = model_data.groupby(["layer", "head"])["bos_fraction"].mean().reset_index()

    for _, row in avg_bos.iterrows():
        bos_records.append(
            {
                "model": model,
                "layer": int(row["layer"]),
                "head": int(row["head"]),
                "bos_fraction": float(row["bos_fraction"]),
            }
        )

df_bos = pd.DataFrame(bos_records)
save_analysis(df_bos, "04_bos_attention.csv")

# Identify strong BOS sink heads (> 0.3 mean BOS attention)
bos_threshold = 0.3
for model in MODELS:
    md = df_bos[df_bos["model"] == model]
    if len(md) == 0:
        continue
    strong_bos = md[md["bos_fraction"] > bos_threshold]
    print(f"\n{model}: {len(strong_bos)} heads with BOS fraction > {bos_threshold}")
    for _, row in (
        strong_bos.sort_values("bos_fraction", ascending=False).head(5).iterrows()
    ):
        print(
            f"  L{int(row['layer'])}H{int(row['head'])}: BOS fraction = {row['bos_fraction']:.3f}"
        )


pythia-70m: 18 heads with BOS fraction > 0.3
  L4H3: BOS fraction = 0.912
  L5H4: BOS fraction = 0.871
  L4H1: BOS fraction = 0.868
  L5H6: BOS fraction = 0.835
  L5H5: BOS fraction = 0.810

pythia-160m: 86 heads with BOS fraction > 0.3
  L8H6: BOS fraction = 1.000
  L9H11: BOS fraction = 0.999
  L9H10: BOS fraction = 0.993
  L8H9: BOS fraction = 0.991
  L10H3: BOS fraction = 0.991

pythia-410m: 245 heads with BOS fraction > 0.3
  L20H8: BOS fraction = 1.000
  L20H9: BOS fraction = 1.000
  L20H10: BOS fraction = 1.000
  L20H5: BOS fraction = 1.000
  L0H1: BOS fraction = 1.000

pythia-1b: 75 heads with BOS fraction > 0.3
  L0H2: BOS fraction = 0.999
  L1H4: BOS fraction = 0.874
  L10H4: BOS fraction = 0.820
  L13H5: BOS fraction = 0.802
  L12H7: BOS fraction = 0.769

pythia-1.4b: 260 heads with BOS fraction > 0.3
  L17H2: BOS fraction = 1.000
  L16H8: BOS fraction = 1.000
  L13H4: BOS fraction = 1.000
  L16H9: BOS fraction = 1.000
  L0H0: BOS fraction = 1.000


In [19]:
# Visualize: BOS attention heatmap per model (head x layer)
for model in MODELS:
    model_bos = df_bos[df_bos["model"] == model]
    if len(model_bos) == 0:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    pivot = model_bos.pivot(index="head", columns="layer", values="bos_fraction")

    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.8), max(5, n_heads * 0.5)))
    sns.heatmap(
        pivot,
        annot=n_heads <= 16,
        fmt=".2f",
        cmap="Reds",
        square=True,
        linewidths=0,
        linecolor="none",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Head")
    ax.set_title(f"BOS Attention Fraction: {model}")
    save_figure(fig, f"viz_04_07_bos_attention_{model}.png")

In [20]:
# BOS attention across layers (mean over heads)
fig, ax = plt.subplots(figsize=(12, 6))

for model in MODELS:
    model_bos = df_bos[df_bos["model"] == model]
    if len(model_bos) == 0:
        continue
    layer_means = model_bos.groupby("layer")["bos_fraction"].mean()
    ax.plot(
        layer_means.index,
        layer_means.values,
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("Mean BOS Attention Fraction")
ax.set_title("Attention Sink: BOS Fraction Across Layers (avg over heads)")
ax.legend()
ax.axhline(
    y=1.0 / SEQ_LEN_WITH_BOS, color="gray", linestyle="--", alpha=0.3, label="Uniform"
)
save_figure(fig, "viz_04_08_bos_across_layers.png")

## 8. Induction Pattern Analysis: Correct vs Incorrect

For identified induction heads: compare attention distributions for examples where
the model predicts correctly (`logit_lens_prob_correct` at final layer > 0.5) vs
incorrectly (< 0.5). Do induction heads attend more to the induction cue for
correct predictions?

In [21]:
# Identify induction heads from overall classification
induction_heads = {}
for model in MODELS:
    model_roles = df_head_roles[
        (df_head_roles["model"] == model) & (df_head_roles["role"] == "induction")
    ]
    if len(model_roles) > 0:
        induction_heads[model] = list(
            zip(model_roles["layer"].values, model_roles["head"].values)
        )
        print(f"{model}: {len(induction_heads[model])} induction heads")
        for l, h in induction_heads[model]:
            print(f"  L{l}H{h}")
    else:
        induction_heads[model] = []
        print(f"{model}: no induction heads found")

pythia-70m: 3 induction heads
  L0H3
  L0H5
  L2H6
pythia-160m: 5 induction heads
  L0H7
  L1H6
  L1H10
  L3H3
  L6H10
pythia-410m: 8 induction heads
  L0H5
  L1H4
  L2H10
  L6H0
  L7H0
  L7H8
  L9H8
  L9H9
pythia-1b: 4 induction heads
  L1H7
  L4H3
  L5H1
  L10H6
pythia-1.4b: 7 induction heads
  L0H3
  L2H0
  L2H5
  L4H7
  L5H11
  L7H15
  L9H1


In [22]:
# Split examples into correct vs incorrect using logit lens at final layer
induction_analysis_records = []

for model in MODELS:
    if not induction_heads.get(model):
        continue

    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model}:")

    for band in BANDS:
        attn = all_attn.get(model, {}).get("draw_1", {}).get(band)
        prob_correct = all_logit_lens_prob.get(model, {}).get("draw_1", {}).get(band)

        if attn is None or prob_correct is None:
            continue

        # prob_correct shape: (N, n_layers)
        # Use final layer probability to split correct vs incorrect
        final_prob = prob_correct[:, -1]  # (N,)
        correct_mask = final_prob > 0.5
        incorrect_mask = ~correct_mask

        n_correct = correct_mask.sum()
        n_incorrect = incorrect_mask.sum()

        if n_correct < 2 or n_incorrect < 2:
            continue

        for layer, head in induction_heads[model]:
            # Attention pattern for this head: (N, 22)
            head_attn = attn[:, layer, head, :]  # (N, seq_len_with_bos)

            # Induction score for correct vs incorrect examples
            induction_score_correct = head_attn[
                correct_mask, MODEL_INDUCTION_CUE_POS
            ].mean()
            induction_score_incorrect = head_attn[
                incorrect_mask, MODEL_INDUCTION_CUE_POS
            ].mean()

            # Full positional stats for correct/incorrect
            stats_correct = compute_positional_attention_stats(
                attn[correct_mask][:, layer : layer + 1, head : head + 1, :]
            )
            stats_incorrect = compute_positional_attention_stats(
                attn[incorrect_mask][:, layer : layer + 1, head : head + 1, :]
            )

            record = {
                "model": model,
                "band": band,
                "layer": layer,
                "head": head,
                "n_correct": int(n_correct),
                "n_incorrect": int(n_incorrect),
                "induction_score_correct": float(induction_score_correct),
                "induction_score_incorrect": float(induction_score_incorrect),
                "induction_score_diff": float(
                    induction_score_correct - induction_score_incorrect
                ),
            }
            # Add mean positional stats for correct/incorrect
            for stat_name in [
                "bos_fraction",
                "source_fraction",
                "target_fraction",
                "repeat_fraction",
                "entropy",
            ]:
                if stat_name in stats_correct:
                    record[f"{stat_name}_correct"] = float(
                        stats_correct[stat_name].mean()
                    )
                if stat_name in stats_incorrect:
                    record[f"{stat_name}_incorrect"] = float(
                        stats_incorrect[stat_name].mean()
                    )

            induction_analysis_records.append(record)

    # Report
    model_records = [r for r in induction_analysis_records if r["model"] == model]
    if model_records:
        avg_diff = np.mean([r["induction_score_diff"] for r in model_records])
        print(f"  Mean induction score diff (correct - incorrect): {avg_diff:.4f}")

df_induction = pd.DataFrame(induction_analysis_records)
save_analysis(df_induction, "04_induction_correct_vs_incorrect.csv")
print(f"\nInduction analysis records: {len(df_induction)}")


pythia-70m:
  Mean induction score diff (correct - incorrect): 0.0209

pythia-160m:
  Mean induction score diff (correct - incorrect): -0.0049

pythia-410m:
  Mean induction score diff (correct - incorrect): 0.0007

pythia-1b:
  Mean induction score diff (correct - incorrect): -0.0039

pythia-1.4b:
  Mean induction score diff (correct - incorrect): -0.0021

Induction analysis records: 135


In [23]:
# Visualize: induction score for correct vs incorrect, per band, for each model
if len(df_induction) > 0 and "model" in df_induction.columns:
    for model in MODELS:
        model_data = df_induction[df_induction["model"] == model]
        if len(model_data) == 0:
            continue

        n_ind_heads = len(induction_heads.get(model, []))
        if n_ind_heads == 0:
            continue

        fig, axes = plt.subplots(
            1, min(n_ind_heads, 6), figsize=(5 * min(n_ind_heads, 6), 5), sharey=True
        )
        if n_ind_heads == 1:
            axes = [axes]

        for idx, (layer, head) in enumerate(induction_heads[model][:6]):
            ax = axes[idx]
            head_data = model_data[
                (model_data["layer"] == layer) & (model_data["head"] == head)
            ]
            if len(head_data) == 0:
                continue

            bands_present = [b for b in BANDS if b in head_data["band"].values]
            x = np.arange(len(bands_present))
            width = 0.35

            vals_correct = [
                head_data[head_data["band"] == b]["induction_score_correct"].values[0]
                for b in bands_present
            ]
            vals_incorrect = [
                head_data[head_data["band"] == b]["induction_score_incorrect"].values[0]
                for b in bands_present
            ]

            ax.bar(
                x - width / 2,
                vals_correct,
                width,
                label="Correct",
                color="#2ca02c",
                alpha=0.8,
            )
            ax.bar(
                x + width / 2,
                vals_incorrect,
                width,
                label="Incorrect",
                color="#d62728",
                alpha=0.8,
            )

            ax.set_xticks(x)
            ax.set_xticklabels(
                [BAND_NAMES.get(b, b) for b in bands_present],
                rotation=45,
                ha="right",
                fontsize=8,
            )
            ax.set_title(f"L{layer}H{head}")

        axes[0].set_ylabel("Induction Score")
        axes[0].legend(fontsize=8)
        fig.suptitle(f"Induction Score: Correct vs Incorrect: {model}", y=1.02)
        fig.tight_layout()
        save_figure(fig, f"viz_04_09_induction_correct_vs_incorrect_{model}.png")
else:
    print("No induction analysis records: skipping visualization.")

In [24]:
# Attention distribution comparison for induction heads: correct vs incorrect
# Show full positional distribution side by side for one example model
example_model = "pythia-410m" if "pythia-410m" in MODELS else MODELS[0]

if induction_heads.get(example_model):
    # Use first induction head, first band, draw_1
    layer, head = induction_heads[example_model][0]
    band = BANDS[0]

    attn = all_attn.get(example_model, {}).get("draw_1", {}).get(band)
    prob_correct = (
        all_logit_lens_prob.get(example_model, {}).get("draw_1", {}).get(band)
    )

    if attn is not None and prob_correct is not None:
        final_prob = prob_correct[:, -1]
        correct_mask = final_prob > 0.5
        incorrect_mask = ~correct_mask

        head_attn = attn[:, layer, head, :]

        fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

        position_labels = (
            ["BOS"]
            + [f"S{i + 1}" for i in range(5)]
            + ["T"]
            + [f"D{i + 1}" for i in range(10)]
            + [f"S'{i + 1}" for i in range(5)]
        )
        x = np.arange(SEQ_LEN_WITH_BOS)

        # Correct examples
        if correct_mask.sum() > 0:
            mean_attn_correct = head_attn[correct_mask].mean(axis=0)
            axes[0].bar(x, mean_attn_correct, color="#2ca02c", alpha=0.8)
            axes[0].set_title(f"Correct (n={correct_mask.sum()})")

        # Incorrect examples
        if incorrect_mask.sum() > 0:
            mean_attn_incorrect = head_attn[incorrect_mask].mean(axis=0)
            axes[1].bar(x, mean_attn_incorrect, color="#d62728", alpha=0.8)
            axes[1].set_title(f"Incorrect (n={incorrect_mask.sum()})")

        for ax in axes:
            ax.set_xticks(x)
            ax.set_xticklabels(position_labels, rotation=45, ha="right", fontsize=7)
            ax.set_xlabel("Position")
            # Highlight induction cue position
            ax.axvline(
                x=MODEL_INDUCTION_CUE_POS,
                color="orange",
                linestyle="--",
                alpha=0.5,
                label="Induction cue",
            )

        axes[0].set_ylabel("Mean Attention Weight")
        axes[0].legend(fontsize=8)
        fig.suptitle(
            f"Attention Distribution L{layer}H{head}: {example_model}, "
            f"{BAND_NAMES.get(band, band)}",
            y=1.02,
        )
        fig.tight_layout()
        save_figure(fig, f"viz_04_10_induction_attn_distribution_{example_model}.png")

## 9. Cross-Model Comparison

Compare attention characteristics across model sizes:
- Number and fraction of induction heads
- Mean induction score
- Mean entropy
- Head specialization (fraction of non-diffuse heads)
- Copy score correlation with induction score

In [25]:
# Build cross-model comparison DataFrame
cross_model_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]
    n_total_heads = n_layers * n_heads

    model_roles = df_head_roles[df_head_roles["model"] == model]
    if len(model_roles) == 0:
        continue

    # Role counts
    role_counts = model_roles["role"].value_counts()
    n_induction = int(role_counts.get("induction", 0))
    n_prev_token = int(role_counts.get("previous_token", 0))
    n_bos_sink = int(role_counts.get("bos_sink", 0))
    n_diffuse = int(role_counts.get("diffuse", 0))

    # Mean attention stats (draw_1, all bands)
    model_pos = df_pos_attn[
        (df_pos_attn["model"] == model) & (df_pos_attn["draw"] == "draw_1")
    ]
    mean_induction = (
        model_pos["induction_score"].mean() if len(model_pos) > 0 else np.nan
    )
    mean_entropy_val = model_pos["entropy"].mean() if len(model_pos) > 0 else np.nan
    mean_bos = model_pos["bos_fraction"].mean() if len(model_pos) > 0 else np.nan

    # Stability
    model_stab = df_stability[df_stability["model"] == model]
    mean_stability = (
        model_stab["majority_fraction"].mean() if len(model_stab) > 0 else np.nan
    )
    pct_stable = 100 * model_stab["is_stable"].mean() if len(model_stab) > 0 else np.nan

    cross_model_records.append(
        {
            "model": model,
            "model_capacity": MODEL_CAPACITY[model],
            "n_layers": n_layers,
            "n_heads": n_heads,
            "n_total_heads": n_total_heads,
            "n_induction": n_induction,
            "pct_induction": 100 * n_induction / n_total_heads,
            "n_prev_token": n_prev_token,
            "n_bos_sink": n_bos_sink,
            "n_diffuse": n_diffuse,
            "pct_specialized": 100 * (1 - n_diffuse / n_total_heads),
            "mean_induction_score": mean_induction,
            "mean_entropy": mean_entropy_val,
            "mean_bos_fraction": mean_bos,
            "pct_stable_heads": pct_stable,
            "mean_majority_fraction": mean_stability,
        }
    )

df_cross_model = pd.DataFrame(cross_model_records)
save_analysis(df_cross_model, "04_cross_model_comparison.csv")

print("Cross-Model Attention Summary:")
display_cols = [
    "model",
    "n_total_heads",
    "n_induction",
    "pct_induction",
    "pct_specialized",
    "mean_entropy",
    "pct_stable_heads",
]
available_cols = [c for c in display_cols if c in df_cross_model.columns]
print(df_cross_model[available_cols].to_string(index=False, float_format="%.2f"))

Cross-Model Attention Summary:
      model  n_total_heads  n_induction  pct_induction  pct_specialized  mean_entropy  pct_stable_heads
 pythia-70m             48            3           6.25            64.58          1.86             77.08
pythia-160m            144            5           3.47            74.31          1.66             82.64
pythia-410m            384            8           2.08            71.09          2.10             86.98
  pythia-1b            128            4           3.12            67.97          2.57             86.72
pythia-1.4b            384            7           1.82            73.44          1.96             87.24


In [26]:
# Visualize: scaling of attention characteristics with model size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ["pct_induction", "pct_specialized", "mean_entropy"]
titles = ["% Induction Heads", "% Specialized Heads", "Mean Entropy (bits)"]

for ax, metric, title in zip(axes, metrics, titles):
    if metric not in df_cross_model.columns:
        continue
    for _, row in df_cross_model.iterrows():
        color = MODEL_COLORS.get(row["model"], "gray")
        ax.scatter(
            row["model_capacity"],
            row[metric],
            color=color,
            s=100,
            zorder=5,
            label=row["model"],
        )

    # Connect with line
    sorted_data = df_cross_model.sort_values("model_capacity")
    ax.plot(
        sorted_data["model_capacity"],
        sorted_data[metric],
        color="gray",
        alpha=0.5,
        linestyle="--",
    )

    ax.set_xlabel("Model Capacity (M params)")
    ax.set_title(title)
    ax.set_xscale("log")

axes[0].legend(fontsize=8)
fig.suptitle("Attention Characteristics vs Model Scale", y=1.02)
fig.tight_layout()
save_figure(fig, "viz_04_11_cross_model_scaling.png")

In [27]:
# Copy score vs induction score scatter (across all heads)
if len(df_copy) > 0 and len(df_head_roles) > 0:
    # Merge copy scores with head roles
    df_merged = df_head_roles.merge(df_copy, on=["model", "layer", "head"], how="inner")

    if len(df_merged) > 0:
        fig, axes = plt.subplots(
            1, len(MODELS), figsize=(5 * len(MODELS), 5), sharey=True
        )
        if len(MODELS) == 1:
            axes = [axes]

        role_color_map = {
            "induction": "#98df8a",
            "previous_token": "#aec7e8",
            "bos_sink": "#ff9896",
            "diffuse": "#d9d9d9",
        }

        for ax, model in zip(axes, MODELS):
            md = df_merged[df_merged["model"] == model]
            if len(md) == 0:
                continue

            for role in ["diffuse", "bos_sink", "previous_token", "induction"]:
                role_data = md[md["role"] == role]
                if len(role_data) == 0:
                    continue
                ax.scatter(
                    role_data["induction_score"],
                    role_data["copy_score"],
                    color=role_color_map.get(role, "gray"),
                    label=role,
                    alpha=0.7,
                    s=30,
                )

            ax.set_xlabel("Induction Score")
            ax.set_title(model)

        axes[0].set_ylabel("Copy Score")
        axes[-1].legend(fontsize=8, loc="upper left")
        fig.suptitle("Copy Score vs Induction Score by Head Role", y=1.02)
        fig.tight_layout()
        save_figure(fig, "viz_04_12_copy_vs_induction.png")
else:
    print("Skipping copy score vs induction analysis (missing data).")

In [28]:
# Induction score by band across models
for model in MODELS:
    model_data = df_pos_attn[
        (df_pos_attn["model"] == model) & (df_pos_attn["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    # Only show for identified induction heads
    ind_heads_model = induction_heads.get(model, [])
    if not ind_heads_model:
        continue

    ind_data = model_data[
        model_data.apply(lambda r: (r["layer"], r["head"]) in ind_heads_model, axis=1)
    ]
    if len(ind_data) == 0:
        continue

    fig = plot_boxplot_by_band(
        ind_data,
        x="band",
        y="induction_score",
        title=f"Induction Score by Band (Induction Heads Only): {model}",
        ylabel="Induction Score",
    )
    save_figure(fig, f"viz_04_13_induction_by_band_{model}.png")

LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


## 10. Summary

Consolidate all attention analysis results and key findings.

In [29]:
# Master attention summary
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    for draw in DRAWS:
        for band in BANDS:
            # Positional attention stats (averaged over all heads)
            pos_data = df_pos_attn[
                (df_pos_attn["model"] == model)
                & (df_pos_attn["draw"] == draw)
                & (df_pos_attn["band"] == band)
            ]
            if len(pos_data) == 0:
                continue

            # FREQUENCY_RANK maps 'control' -> None; convert to NaN for DataFrame
            freq_rank = FREQUENCY_RANK.get(band)
            record = {
                "model": model,
                "draw": draw,
                "band": band,
                "model_capacity": MODEL_CAPACITY[model],
                "frequency_rank": freq_rank if freq_rank is not None else np.nan,
                "mean_induction_score": pos_data["induction_score"].mean(),
                "mean_bos_fraction": pos_data["bos_fraction"].mean(),
                "mean_target_fraction": pos_data["target_fraction"].mean(),
                "mean_repeat_fraction": pos_data["repeat_fraction"].mean(),
                "mean_source_fraction": pos_data["source_fraction"].mean(),
                "mean_entropy": pos_data["entropy"].mean(),
            }

            # Entropy data
            ent_data = df_entropy[
                (df_entropy["model"] == model)
                & (df_entropy["draw"] == draw)
                & (df_entropy["band"] == band)
            ]
            if len(ent_data) > 0:
                record["entropy_median"] = ent_data["entropy_median"].mean()

            master_records.append(record)

df_master = pd.DataFrame(master_records)
save_analysis(df_master, "04_master_attention_summary.csv")
print(f"Master attention summary: {len(df_master)} records")
print(
    df_master.groupby("model")[
        ["mean_induction_score", "mean_bos_fraction", "mean_entropy"]
    ]
    .mean()
    .round(4)
)

Master attention summary: 75 records
             mean_induction_score  mean_bos_fraction  mean_entropy
model                                                             
pythia-1.4b                0.0286             0.4937        1.9531
pythia-160m                0.0326             0.3759        1.6614
pythia-1b                  0.0396             0.3563        2.5692
pythia-410m                0.0333             0.4635        2.0980
pythia-70m                 0.0525             0.2914        1.8596


In [30]:
# Final summary heatmaps: mean induction score (model x band)
summary_pivot = (
    df_master.groupby(["model", "band"])["mean_induction_score"].mean().unstack()
)
summary_pivot = summary_pivot.reindex(index=MODELS, columns=BANDS)

fig = plot_metric_heatmap(
    summary_pivot,
    title="Mean Induction Score (Model x Band)",
    xlabel="Band",
    ylabel="Model",
    fmt=".4f",
    cmap="YlOrRd",
)
save_figure(fig, "viz_04_14_induction_score_heatmap.png")

In [31]:
# Entropy heatmap (model x band)
entropy_pivot = df_master.groupby(["model", "band"])["mean_entropy"].mean().unstack()
entropy_pivot = entropy_pivot.reindex(index=MODELS, columns=BANDS)

fig = plot_metric_heatmap(
    entropy_pivot,
    title="Mean Attention Entropy (Model x Band)",
    xlabel="Band",
    ylabel="Model",
    fmt=".2f",
    cmap="viridis",
)
save_figure(fig, "viz_04_15_entropy_heatmap.png")

In [32]:
print("\n" + "=" * 70)
print("NOTEBOOK 04: ATTENTION ANALYSIS COMPLETE")
print("=" * 70)

print(f"\nOutput CSVs in: {ANALYSIS_DIR}")
for f in sorted(ANALYSIS_DIR.glob("04_*")):
    print(f"  {f.name}")

print(f"\nFigures in: {VIZ_DIR}")
for f in sorted(VIZ_DIR.glob("viz_04_*")):
    print(f"  {f.name}")

print("\n--- Key Findings ---")
for model in MODELS:
    cm = df_cross_model[df_cross_model["model"] == model]
    if len(cm) == 0:
        continue
    row = cm.iloc[0]
    print(f"\n{model}:")
    print(f"  Total heads: {int(row['n_total_heads'])}")
    print(f"  Induction heads: {int(row['n_induction'])} ({row['pct_induction']:.1f}%)")
    print(f"  Specialized (non-diffuse): {row['pct_specialized']:.1f}%")
    print(f"  Mean entropy: {row['mean_entropy']:.2f} bits")
    print(f"  Band-stable heads: {row['pct_stable_heads']:.1f}%")


NOTEBOOK 04: ATTENTION ANALYSIS COMPLETE

Output CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/base/analysis
  04_attention_entropy.csv
  04_bos_attention.csv
  04_copy_scores.csv
  04_cross_model_comparison.csv
  04_head_role_by_band.csv
  04_head_role_classification.csv
  04_head_role_stability.csv
  04_head_role_summary.csv
  04_induction_correct_vs_incorrect.csv
  04_master_attention_summary.csv
  04_positional_attention_stats.csv

Figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/attention/base/viz
  viz_04_01_positional_attention_pythia-1.4b.png
  viz_04_01_positional_attention_pythia-160m.png
  viz_04_01_positional_attention_pythia-1b.png
  viz_04_01_positional_attention_pythia-410m.png
  viz_04_01_positional_attention_pythia-70m.png
  viz_04_02_entropy_heatmap_pythia-1.4b.png
  viz_04_02_entropy_heatmap_pythia-160m.png
  viz_04_02_entropy_heatmap_pythia-1b.png
  viz_04_02_entropy_heatmap_pythia-410m.png
  viz_04_02_entropy_heatmap_p